In [1]:
import os, time, json
import numpy as np
import pandas as pd
import torch
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

DATA_DIR = "/kaggle/input/datasets/alessandrasala79/ai-vs-human-generated-dataset"
df = pd.read_csv(f"{DATA_DIR}/train.csv")[["file_name", "label"]]

# In this dataset: 1 = AI, 0 = real (confirmed visually)
FAKE, REAL = 1, 0

# Rows come in pairs (0-1, 2-3, ...): one real photo + its AI twin
df["pair_id"] = np.arange(len(df)) // 2

# Split by PAIR so twins always land in the same split (prevents leakage)
rng = np.random.default_rng(42)
pairs = rng.permutation(df.pair_id.unique())
n_train, n_val = int(0.8 * len(pairs)), int(0.1 * len(pairs))
split_of = {}
for p in pairs[:n_train]:                 split_of[p] = "train"
for p in pairs[n_train:n_train + n_val]:  split_of[p] = "val"
for p in pairs[n_train + n_val:]:         split_of[p] = "test"
df["split"] = df.pair_id.map(split_of)

train_df = df[df.split == "train"].reset_index(drop=True)
val_df   = df[df.split == "val"].reset_index(drop=True)
test_df  = df[df.split == "test"].reset_index(drop=True)
print(f"Train {len(train_df)} | Val {len(val_df)} | Test {len(test_df)}")
print("Test balance:", test_df.label.value_counts().to_dict())

# Safety check: no pair appears in two splits
assert len(set(train_df.pair_id) & set(test_df.pair_id)) == 0
assert len(set(train_df.pair_id) & set(val_df.pair_id)) == 0
df.to_csv("/kaggle/working/split_v2.csv", index=False)   # record of exactly which image went where

Device: cuda Tesla T4
Train 63960 | Val 7994 | Test 7996
Test balance: {1: 3998, 0: 3998}


In [2]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

IMG_SIZE, BATCH_SIZE = 224, 64
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# Same recipe as v1 AND as the API: only the data changes in v2
tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class CsvImageDataset(Dataset):
    """Reads labels from a table instead of folder names."""
    def __init__(self, frame, transform):
        self.paths = frame.file_name.tolist()
        self.labels = frame.label.tolist()
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(f"{DATA_DIR}/{self.paths[i]}").convert("RGB")
        return self.transform(img), self.labels[i]

def make_loader(frame, shuffle):
    return DataLoader(CsvImageDataset(frame, tf), batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=4, pin_memory=True)

train_loader = make_loader(train_df, shuffle=True)
val_loader   = make_loader(val_df,   shuffle=False)
test_loader  = make_loader(test_df,  shuffle=False)

images, labels = next(iter(train_loader))
print("Batch shape:", images.shape)
print("AI in batch:", (labels == FAKE).sum().item(), "| Real:", (labels == REAL).sum().item())

Batch shape: torch.Size([64, 3, 224, 224])
AI in batch: 34 | Real: 30


In [3]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
LEARNING_RATE = 1e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
print("Weights:", f"{sum(p.numel() for p in model.parameters()):,}")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 194MB/s]

Weights: 4,010,110


In [4]:
from torch.amp import autocast, GradScaler

EPOCHS = 3
SAVE_PATH = "/kaggle/working/verify_effnetb0_v2.pth"
scaler = GradScaler("cuda")

def evaluate(loader):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with autocast("cuda"):
                out = model(x)
                loss = criterion(out, y)
            preds = out.argmax(1)
            loss_sum += loss.item() * y.size(0)
            correct += (preds == y).sum().item()
            total += y.size(0)
            all_preds.append(preds.cpu()); all_labels.append(y.cpu())
    return loss_sum / total, correct / total, torch.cat(all_preds), torch.cat(all_labels)

best_val_acc, history = 0.0, []
for epoch in range(EPOCHS):
    model.train()
    t0 = time.time()
    run_loss, run_correct, seen = 0.0, 0, 0
    for i, (x, y) in enumerate(train_loader):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad()
        with autocast("cuda"):
            out = model(x)
            loss = criterion(out, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        run_loss += loss.item() * y.size(0)
        run_correct += (out.argmax(1) == y).sum().item()
        seen += y.size(0)
        if (i + 1) % 200 == 0:
            print(f"  epoch {epoch+1} | batch {i+1}/{len(train_loader)} | "
                  f"loss {run_loss/seen:.4f} | acc {run_correct/seen:.4f}")

    val_loss, val_acc, _, _ = evaluate(val_loader)
    mins = (time.time() - t0) / 60
    print(f"EPOCH {epoch+1}: train acc {run_correct/seen:.4f} | val acc {val_acc:.4f} | "
          f"val loss {val_loss:.4f} | {mins:.1f} min")
    history.append({"epoch": epoch + 1, "train_loss": run_loss / seen, "train_acc": run_correct / seen,
                    "val_loss": val_loss, "val_acc": val_acc, "minutes": mins})
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  -> new best model saved ({val_acc:.4f})")

  epoch 1 | batch 200/1000 | loss 0.1770 | acc 0.9357
  epoch 1 | batch 400/1000 | loss 0.1072 | acc 0.9612
  epoch 1 | batch 600/1000 | loss 0.0794 | acc 0.9713
  epoch 1 | batch 800/1000 | loss 0.0654 | acc 0.9765
  epoch 1 | batch 1000/1000 | loss 0.0557 | acc 0.9801
EPOCH 1: train acc 0.9801 | val acc 0.9966 | val loss 0.0089 | 5.2 min
  -> new best model saved (0.9966)
  epoch 2 | batch 200/1000 | loss 0.0067 | acc 0.9983
  epoch 2 | batch 400/1000 | loss 0.0066 | acc 0.9982
  epoch 2 | batch 600/1000 | loss 0.0067 | acc 0.9981
  epoch 2 | batch 800/1000 | loss 0.0076 | acc 0.9977
  epoch 2 | batch 1000/1000 | loss 0.0074 | acc 0.9976
EPOCH 2: train acc 0.9976 | val acc 0.9980 | val loss 0.0056 | 4.0 min
  -> new best model saved (0.9980)
  epoch 3 | batch 200/1000 | loss 0.0042 | acc 0.9987
  epoch 3 | batch 400/1000 | loss 0.0048 | acc 0.9985
  epoch 3 | batch 600/1000 | loss 0.0044 | acc 0.9986
  epoch 3 | batch 800/1000 | loss 0.0043 | acc 0.9985
  epoch 3 | batch 1000/1000 | 

In [5]:
model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
test_loss, test_acc, preds, labels = evaluate(test_loader)

# Careful: in v2, FAKE = 1 and REAL = 0
fake_caught  = ((preds == FAKE) & (labels == FAKE)).sum().item()
fake_missed  = ((preds == REAL) & (labels == FAKE)).sum().item()
real_correct = ((preds == REAL) & (labels == REAL)).sum().item()
real_flagged = ((preds == FAKE) & (labels == REAL)).sum().item()

false_positive_rate = real_flagged / (real_flagged + real_correct)   # real photos called AI
miss_rate           = fake_missed / (fake_missed + fake_caught)       # AI images called real

print(f"TEST ACCURACY: {test_acc:.4f}")
print(f"AI caught {fake_caught} | AI missed {fake_missed}")
print(f"Real correct {real_correct} | Real wrongly flagged {real_flagged}")
print(f"False positive rate: {false_positive_rate:.4f} | Miss rate: {miss_rate:.4f}")

results = {
    "dataset": "alessandrasala79/ai-vs-human-generated-dataset",
    "split": "by pair, 80/10/10, seed 42",
    "split_sizes": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
    "test_accuracy": test_acc, "false_positive_rate": false_positive_rate, "miss_rate": miss_rate,
    "best_val_acc": best_val_acc, "history": history,
    "class_to_idx": {"FAKE": FAKE, "REAL": REAL},
    "img_size": IMG_SIZE, "mean": MEAN, "std": STD,
    "learning_rate": LEARNING_RATE, "epochs": EPOCHS,
}
with open("/kaggle/working/results_v2.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved results_v2.json")

TEST ACCURACY: 0.9984
AI caught 3994 | AI missed 4
Real correct 3989 | Real wrongly flagged 9
False positive rate: 0.0023 | Miss rate: 0.0010
Saved results_v2.json
